In [2]:
import sys
!{sys.executable} -m pip install -q yfinance pandas-datareader scikit-learn xgboost lightgbm shap arch hmmlearn pykalman xlsxwriter imbalanced-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.1/252.1 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 10.9 MB/s eta 0:00:00


In [3]:
import sys
!{sys.executable} -m pip install -q imblearn

In [4]:
from imblearn.over_sampling import SMOTE

In [5]:
import os, re, time, json, warnings
from pathlib import Path
from typing import List, Dict
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader as pdr
import shap
from arch import arch_model

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from hmmlearn import hmm as hmmlib
from pykalman import KalmanFilter

OUTPUT_DIR = Path("/content/outputs_v29_egarch_spx_amplitude")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
MIN_DATA_START = pd.Timestamp("2000-01-01")
TRAIN_START    = "2000-01-01"

# Split 80/20 chronologique — TEST_DATE calculé dynamiquement après chargement
TEST_DATE = None  # sera défini après chargement : 80e percentile temporel

YF_CHUNK_SIZE        = 40
SLEEP_BETWEEN_CHUNKS = 1.0
MIN_COLUMN_COVERAGE  = 0.90
MAX_FIRST_VALID_LAG_DAYS = 365
ROLLING_QUANTILE_WINDOW  = 504

FLAT_THRESHOLD_ABS = 0.003  # jours flat supprimés

CLASS_QUANTILES = [0.25, 0.75]
CLASS_LABELS    = ["DOWN_FORT","DOWN_FAIBLE","UP_FAIBLE","UP_FORT"]

HORIZONS_TO_TEST = [1, 3, 5]
ALL_REGIMES      = ["CALM","NORMAL","STRESS","GLOBAL"]

SHAP_PILOT_N_ESTIMATORS  = 150
SHAP_TOP_BASE_N          = 40
SHAP_TOP_FINAL_N         = 30
N_TOP_FOR_INTERACTIONS   = 20
INTERACTION_ROLLING_WINDOW = 20

MIN_N_FEATURES = 5
MAX_N_FEATURES = 30

# EGARCH SPX
EGARCH_HORIZONS = [1, 3, 5]  # horizons de forecast variance SPX

np.random.seed(RANDOM_STATE)


In [6]:
# (select_and_filter_features est definie dans la cellule suivante - cette
# cellule etait un doublon residuel, neutralisee pour eviter la confusion.)


In [7]:
from typing import List, Dict
# =============================================================================
# Sélection de features par corrélation (reprise à l'identique du notebook
# original) : produit une LISTE ORDONNÉE de features (les plus corrélées au
# target en premier, en excluant les features trop corrélées entre elles).
# Cette liste classée sert ensuite de base pour tester N=1..20 features.
# =============================================================================

correlation_threshold_target = 0.05
correlation_threshold_features = 0.9

def select_and_filter_features(
    df: pd.DataFrame,
    all_features: List[str],
    target_col: str = "VIX_Direction",
    correlation_threshold_target: float = 0.05,
    correlation_threshold_features: float = 0.9,
    max_features_to_select: int = 20
) -> List[str]:
    cols_to_use = [f for f in all_features if f in df.columns] + [target_col]
    df_for_corr = df[cols_to_use].copy().dropna()

    if df_for_corr.empty:
        warnings.warn("DataFrame for correlation is empty after dropping NaNs.")
        return []

    correlations = df_for_corr.corr(method='spearman')[target_col].abs().sort_values(ascending=False)

    selected_features_initial = correlations[correlations >= correlation_threshold_target].index.tolist()
    if target_col in selected_features_initial:
        selected_features_initial.remove(target_col)

    if not selected_features_initial:
        return []

    features_small = []
    high_corr_features = list(selected_features_initial)
    df_for_inter_corr = df_for_corr[[f for f in high_corr_features if f in df_for_corr.columns]].copy()

    while len(high_corr_features) > 0 and len(features_small) < max_features_to_select:
        current_correlations = correlations[high_corr_features]
        if current_correlations.empty:
            break
        f = current_correlations.idxmax()
        features_small.append(f)
        high_corr_features.remove(f)

        correlated_with_f = []
        if f in df_for_inter_corr.columns:
            for f_other in high_corr_features:
                if f_other in df_for_inter_corr.columns:
                    try:
                        corr_value = df_for_inter_corr[[f, f_other]].corr(method='spearman').iloc[0, 1]
                        if abs(corr_value) >= correlation_threshold_features:
                            correlated_with_f.append(f_other)
                    except Exception:
                        pass

        for f_to_remove in correlated_with_f:
            if f_to_remove in high_corr_features:
                high_corr_features.remove(f_to_remove)

    return features_small


In [8]:
# =============================================================================
# RECOMMANDATION 2 : génération systématique d'interactions explicites.
#
# Pour un ensemble de features de base (typiquement le top 30 actuel d'un
# régime/horizon), génère TOUTES les paires (ratio + différence) :
#   - ratio      : feat_i / feat_j  (nommé "{i}_div_{j}")
#   - différence : feat_i - feat_j  (nommé "{i}_minus_{j}")
#
# Pour 30 features : 30*29/2 = 435 paires non-ordonnées x 2 opérations = 870
# nouvelles colonnes potentielles. Toutes ne seront pas utiles - elles sont
# ensuite repassées dans select_and_filter_features (même filtre Spearman +
# anti-redondance) pour ne garder que celles qui apportent un signal réel.
#
# Protection : ratio i/j est mis à NaN si |j| < epsilon (évite divisions
# explosives qui pollueraient la sélection avec du bruit numérique).
# =============================================================================

def generate_pairwise_interactions(df: pd.DataFrame, base_features: list,
                                    epsilon: float = 1e-8) -> pd.DataFrame:
    """Génère ratio et différence pour toutes les paires non-ordonnées de
    base_features. Retourne un NOUVEAU dataframe (les colonnes générées
    uniquement, même index que df) - à concaténer par l'appelant."""
    n = len(base_features)
    print(f"[INTERACTIONS] Génération de paires pour {n} features de base "
          f"({n*(n-1)//2} paires x 2 opérations = {n*(n-1)} colonnes max)...")

    new_cols = {}
    for i in range(n):
        for j in range(i + 1, n):
            fi, fj = base_features[i], base_features[j]
            if fi not in df.columns or fj not in df.columns:
                continue

            col_i = df[fi]
            col_j = df[fj]

            # Différence
            diff_name = f"{fi}_minus_{fj}"
            new_cols[diff_name] = col_i - col_j

            # Ratio (protégé contre division par ~0)
            ratio_name = f"{fi}_div_{fj}"
            safe_denom = col_j.where(col_j.abs() >= epsilon, np.nan)
            new_cols[ratio_name] = col_i / safe_denom

    interactions_df = pd.DataFrame(new_cols, index=df.index)
    interactions_df = interactions_df.replace([np.inf, -np.inf], np.nan)
    print(f"[INTERACTIONS] {interactions_df.shape[1]} colonnes d'interactions générées.")
    return interactions_df


In [9]:
BAD_TICKERS = {
    "XXIV",
    "TVIX",
    "ZIV",
    "^MIB",
    "CELG",
    "AET",
    "HES",
    "GPS",
    "JWN",
    "DFS",
    "SPX",
    "EON",
    "EDF",
    "RWE",
    "SZR",
    "ICN",
    "CEIX",
    "MXEA",
    "SQ",
    "SHELL",
    "K",
}

MANUAL_YF_NAMES = {
    "^GSPC": "SP500_Price",
    "^IXIC": "NASDAQ_Price",
    "^DJI": "DOW_Price",
    "^RUT": "Russell_Price",
    "^VIX": "VIX_Price",
    "^VXN": "VXN_NASDAQ_Vol",
    "^OVX": "OVX_Oil_Vol",
    "^GVZ": "GVZ_Gold_Vol",
    "^EVZ": "EVZ_EUR_Vol",
    "^FTSE": "FTSE_UK",
    "^N225": "Nikkei_Japan",
    "^HSI": "HangSeng_HK",
    "^GDAXI": "DAX_Germany",
    "^FCHI": "CAC40_France",
    "^STOXX50E": "STOXX50E_EU",
    "SPY": "SPY",
    "QQQ": "QQQ",
    "TLT": "TLT_LongBond",
    "GLD": "GLD_Gold",
    "USO": "USO_Oil",
    "UUP": "UUP_Dollar",
    "FXE": "FXE_Euro",
    "FXY": "FXY_Yen",
    "HYG": "HYG_HighYield",
    "LQD": "LQD_InvGrade",

    # --- Tickers ajoutés (issus du dictionnaire massif fourni) ---
    "^BVSP": "BOVESPA_Brazil",
    "^AXJO": "ASX_Australia",
    "^AORD": "AORD_AUS",
    "^IBEX": "IBEX_Spain",
    "VXX": "VXX",
    "UVXY": "UVXY",
    "VIXY": "VIXY",
    "SVXY": "SVXY",
    "VXZ": "VXZ",
    "VIXM": "VIXM",
    "XLK": "XLK_Tech",
    "XLF": "XLF_Fin",
    "XLE": "XLE_Energy",
    "XLV": "XLV_Health",
    "XLU": "XLU_Util",
    "XLP": "XLP_Staples",
    "XLI": "XLI_Indust",
    "XLY": "XLY_Disc",
    "XLRE": "XLRE_RE",
    "XLB": "XLB_Materials",
    "XLC": "XLC_CommServ",
    "GOOGL": "GOOGL_Google",
    "META": "META_Meta",
    "AVGO": "AVGO_Broadcom",
    "ASML": "ASML_ASML",
    "WFC": "WFC_WellsFargo",
    "GS": "GS_GoldmanSachs",
    "BLK": "BLK_BlackRock",
    "SCHW": "SCHW_Schwab",
    "MS": "MS_MorganStanley",
    "COF": "COF_CapitalOne",
    "BAX": "BAX_BankBoston",
    "AXP": "AXP_Amex",
    "EQR": "EQR_Equity",
    "PLD": "PLD_Prologis",
    "AMT": "AMT_AmericanTower",
    "EQIX": "EQIX_Equinix",
    "CCI": "CCI_CrownCastle",
    "PSA": "PSA_PublicStorage",
    "ABBV": "ABBV_AbbVie",
    "MRK": "MRK_Merck",
    "BMY": "BMY_BristolMyers",
    "AMGN": "AMGN_Amgen",
    "GILD": "GILD_Gilead",
    "BNTX": "BNTX_BioNTech",
    "MRNA": "MRNA_Moderna",
    "CRSP": "CRSP_CrisprTherapy",
    "VRTX": "VRTX_VertexPharm",
    "ILMN": "ILMN_Illumina",
    "DXCM": "DXCM_Dexcom",
    "TDOC": "TDOC_Teladoc",
    "CI": "CI_Cigna",
    "HUM": "HUM_Humana",
    "RTX": "RTX_Raytheon",
    "LMT": "LMT_LockheedMartin",
    "NOC": "NOC_Northrop",
    "GD": "GD_GeneralDynamics",
    "CAT": "CAT_Caterpillar",
    "DE": "DE_Deere",
    "ITT": "ITT_ITTInc",
    "PAYX": "PAYX_Paychex",
    "CTAS": "CTAS_Cintas",
    "MMM": "3M",
    "HON": "HON_Honeywell",
    "ETN": "ETN_Eaton",
    "EMR": "EMR_Emerson",
    "OTIS": "OTIS_Otis",
    "JCI": "JCI_JohnsonControls",
    "PTC": "PTC_PTC",
    "SMCI": "SMCI_SuperMicroComputer",
    "COP": "COP_ConocoPhillips",
    "SLB": "SLB_Schlumberger",
    "EOG": "EOG_EOGResources",
    "MPC": "MPC_MarathonPetroleum",
    "PSX": "PSX_PhillipsLiquids",
    "VLO": "VLO_Valero",
    "PM": "PM_PhilipMorris",
    "MO": "MO_AltriaMG",
    "BTI": "BTI_BritishAmerican",
    "TBP": "TBP_Tata",
    "BP": "BP_BritishPetroleum",
    "TTE": "TTE_TotalEnergies",
    "ENB": "ENB_EnbridgeInc",
    "MET": "MET_MetalexEnergy",
    "ADM": "ADM_ArcherDaniels",
    "MKC": "MKC_McCormick",
    "SJM": "SJM_JM_Smucker",
    "CPB": "CPB_CampbellSoup",
    "GIS": "GIS_GeneralMills",
    "MDLZ": "MDLZ_Mondelez",
    "NSRGY": "NSRGY_Nestle",
    "TAP": "TAP_MolsonCoors",
    "BDX": "BDX_Becton_Dickinson",
    "CLX": "CLX_Clorox",
    "CL": "CL_Colgate",
    "UL": "UL_Unilever",
    "LVRK": "LVRK_Lavazza",
    "YUM": "YUM_YumBrands",
    "QSR": "QSR_RestaurantBrands",
    "DPZ": "DPZ_Dominos",
    "BLMN": "BLMN_BloombergME",
    "NWL": "NWL_Newell",
    "RRR": "RRR_RareMedica",
    "DASH": "DASH_DoorDash",
    "LYFT": "LYFT_Lyft",
    "UBER": "UBER_Uber",
    "TGT": "TGT_Target",
    "M": "M_Macys",
    "LOW": "LOW_Lowes",
    "ROST": "ROST_RossStores",
    "BBY": "BBY_BestBuy",
    "NEE": "NEE_NextEra",
    "DUK": "DUK_Duke",
    "SO": "SO_SouthernCo",
    "AEP": "AEP_AmericanElectric",
    "EXC": "EXC_Exelon",
    "SRE": "SRE_Sempra",
    "ES": "ES_Evergy",
    "XEL": "XEL_Xcel",
    "PPL": "PPL_PPL",
    "TMUS": "TMUS_TMobileUS",
    "CHTR": "CHTR_Charter",
    "VOD": "VOD_Vodafone",
    "TM": "TM_Telephone",
    "LOGI": "LOGI_Logitech",
    "NET": "NET_Cloudflare",
    "DDOG": "DDOG_Datadog",
    "SLV": "SLV_Silver",
    "UNG": "UNG_Gas",
    "DBC": "DBC_Commodity",
    "DBA": "DBA_Agri",
    "GDX": "GDX_GoldMiners",
    "GDXJ": "GDXJ_JrMiners",
    "PDBC": "PDBC_Commodity2",
    "CORN": "CORN_Corn",
    "SOYB": "SOYB_Soybean",
    "CBOT_W": "Wheat",
    "IEF": "IEF_MidBond",
    "SHY": "SHY_ShortBond",
    "SHV": "SHV_TBill",
    "BIL": "BIL_TBill3M",
    "AGG": "AGG_Aggregate",
    "BND": "BND_TotalBond",
    "JNK": "JNK_HY2",
    "VCIT": "VCIT_CorpIG",
    "VCSH": "VCSH_CorpST",
    "EMB": "EMB_EM",
    "MBB": "MBB_Mortgage",
    "TIP": "TIP_TIPS",
    "BNDX": "BNDX_IntlBond",
    "HYLD": "HYLD_HYieldETF",
    "PFFA": "PFFA_PreferredA",
    "EWJ": "EWJ_Japan",
    "EWG": "EWG_Germany",
    "EWU": "EWU_UK",
    "EWA": "EWA_Australia",
    "EWH": "EWH_HongKong",
    "EWL": "EWL_Switzerland",
    "EWP": "EWP_Spain",
    "EWI": "EWI_Italy",
    "EWQ": "EWQ_France",
    "EWT": "EWT_Taiwan",
    "EWY": "EWY_Korea",
    "EWZ": "EWZ_Brazil",
    "EWC": "EWC_Canada",
    "EWS": "EWS_Singapore",
    "EWM": "EWM_Malaysia",
    "FXI": "FXI_China",
    "MCHI": "MCHI_China2",
    "IEMG": "IEMG_EM",
    "EEM": "EEM_EM2",
    "VEA": "VEA_DM",
    "INDA": "INDA_India",
    "EPI": "EPI_India2",
    "ASHR": "ASHR_China_A",
    "TUR": "TUR_Turkey",
    "EIDO": "EIDO_Indonesia",
    "EPOL": "EPOL_Poland",
    "EZA": "EZA_SouthAfrica",
    "GXG": "GXG_Germany2",
    "EGRX": "EGRX_Greece",
    "FXB": "FXB_GBP",
    "FXA": "FXA_AUD",
    "FXC": "FXC_CAD",
    "FXF": "FXF_CHF",
    "CEW": "CEW_EM_FX",
    "CYB": "CYB_ChineseYuan",
    "BZF": "BZF_BrazilReal",
    "FXD": "FXD_SwedishKrona",
    "FXN": "FXN_NorwegianKrone",
    "GBTC": "GBTC_Bitcoin",
    "IBIT": "IBIT_Bitcoin2",
    "COIN": "COIN_Crypto",
    "MSTR": "MSTR_Bitcoin3",
    "BITO": "BITO_BitcoinETF",
    "ETHA": "ETHA_EthereumETF",
    "MARA": "MARA_Marathon",
    "RIOT": "RIOT_Riot",
    "CLSK": "CLSK_CleanSpark",
    "CIFR": "CIFR_Cipher",
    "CORZ": "CORZ_Core_Sci",
    "IWM": "IWM_SmallCap",
    "IVV": "IVV_SP500",
    "VTI": "VTI_Total",
    "VOO": "VOO_SP500_2",
    "VV": "VV_LargeCap",
    "VTV": "VTV_Value",
    "VUG": "VUG_Growth",
    "VB": "VB_SmallCap2",
    "SCHD": "SCHD_Div",
    "VIG": "VIG_DivGrowth",
    "HDV": "HDV_HighDiv",
    "NOBL": "NOBL_Aristocrat",
    "DGRO": "DGRO_DividendGrowth",
    "QUAL": "QUAL_Quality",
    "VLUE": "VLUE_Value",
    "VYMI": "VYMI_HighDivYield",
    "JEPI": "JEPI_EquityPremiumIncome",
    "XYLD": "XYLD_XYieldETF",
    "QYLD": "QYLD_NasdaqYield",
    "RYLD": "RYLD_Russell2000Yield",
    "ARKK": "ARKK_Innovation",
    "XBI": "XBI_Biotech",
    "SOXX": "SOXX_Semis",
    "IBB": "IBB_Biotech2",
    "IYT": "IYT_Transport",
    "XHB": "XHB_Homebuilders",
    "KRE": "KRE_RegionalBanks",
    "KBE": "KBE_Banks",
    "ITA": "ITA_Defense",
    "XOP": "XOP_OilExploration",
    "OIH": "OIH_OilServices",
    "IYM": "IYM_BasicMaterials",
    "PCAR": "PCAR_PaccarInc",
    "DAL": "DAL_Delta",
    "AAL": "AAL_AmericanAir",
    "UAL": "UAL_UnitedAir",
    "LUV": "LUV_SouthwestAir",
    "ICLN": "ICLN_CleanEnergy",
    "TAN": "TAN_SolarEnergy",
    "MTUM": "MTUM_Momentum",
    "USMV": "USMV_MinVol",
    "SPLV": "SPLV_LowVol_SP500",
    "RSP": "RSP_EqualWeight_SP500",
    "EUSA": "EUSA_EuropeMomentum",
    "EEMV": "EEMV_EMMinVol",
    "VNQ": "VNQ_US_REIT",
    "IYR": "IYR_US_REIT2",
    "REM": "REM_Mortgage_REIT",
    "SPG": "SPG_SimonProperty",
    "AVB": "AVB_AvalonBay",
    "COLD": "COLD_ColdStorage",
    "DLR": "DLR_Digital_Realty",
    "REXR": "REXR_Rexford",
    "HII": "HII_HuntingtonIngalls",
    "L3HARRIS": "L3H_L3Harris",
    "LDOS": "LDOS_LeadosSecurity",
    "EBAY": "EBAY_eBay",
    "MELI": "MELI_MercadoLibre",
    "SHOP": "SHOP_Shopify",
    "SE": "SE_SeaLimited",
    "PDD": "PDD_PinDuoDuo",
    "JD": "JD_JD.com",
    "VIPS": "VIPS_Vipshop",
    "UPST": "UPST_Upstart",
    "RBLX": "RBLX_Roblox",
    "SNOW": "SNOW_Snowflake",
    "CRWD": "CRWD_CrowdStrike",
    "ZM": "ZM_Zoom",
    "ROKU": "ROKU_Roku",
    "PINS": "PINS_Pinterest",
    "SNAP": "SNAP_Snapchat",
    "TERM": "TERM_Terminal",
    "SPCE": "SPCE_VirginGalactic",
}

YF_TICKERS_RAW = """
    ^GSPC ^IXIC ^DJI ^RUT ^VIX ^VXN ^OVX ^GVZ ^EVZ
    ^FTSE ^N225 ^HSI ^GDAXI ^FCHI ^STOXX50E
    SPY QQQ TLT GLD USO UUP FXE FXY HYG LQD
    AAPL MSFT GOOG AMZN NVDA TSLA JPM JNJ V MA PG UNH HD KO PEP T SMFG DIS XOM CVX BAC WMT VZ CSCO ORCL CRM AMD NFLX ADBE INTC CMCSA PFE ABT LLY DHR COST CMG SBUX MCD ACN PYPL QCOM TXN BA GE BABA

    ^BVSP ^AXJO ^AORD ^IBEX VXX UVXY VIXY SVXY VXZ VIXM XLK XLF XLE XLV XLU XLP XLI XLY XLRE
    XLB XLC GOOGL META AVGO ASML WFC GS BLK SCHW MS COF BAX AXP EQR PLD AMT EQIX CCI PSA ABBV
    MRK BMY AMGN GILD BNTX MRNA CRSP VRTX ILMN DXCM TDOC CI HUM RTX LMT NOC GD CAT DE ITT
    PAYX CTAS MMM HON ETN EMR OTIS JCI PTC SMCI COP SLB EOG MPC PSX VLO PM MO BTI TBP BP TTE
    ENB MET ADM MKC SJM CPB GIS MDLZ NSRGY TAP BDX CLX CL UL LVRK YUM QSR DPZ BLMN NWL RRR
    DASH LYFT UBER TGT M LOW ROST BBY NEE DUK SO AEP EXC SRE ES XEL PPL TMUS CHTR VOD TM LOGI
    NET DDOG SLV UNG DBC DBA GDX GDXJ PDBC CORN SOYB CBOT_W IEF SHY SHV BIL AGG BND JNK VCIT
    VCSH EMB MBB TIP BNDX HYLD PFFA EWJ EWG EWU EWA EWH EWL EWP EWI EWQ EWT EWY EWZ EWC EWS
    EWM FXI MCHI IEMG EEM VEA INDA EPI ASHR TUR EIDO EPOL EZA GXG EGRX FXB FXA FXC FXF CEW
    CYB BZF FXD FXN GBTC IBIT COIN MSTR BITO ETHA MARA RIOT CLSK CIFR CORZ IWM IVV VTI VOO VV
    VTV VUG VB SCHD VIG HDV NOBL DGRO QUAL VLUE VYMI JEPI XYLD QYLD RYLD ARKK XBI SOXX IBB
    IYT XHB KRE KBE ITA XOP OIH IYM PCAR DAL AAL UAL LUV ICLN TAN MTUM USMV SPLV RSP EUSA
    EEMV VNQ IYR REM SPG AVB COLD DLR REXR HII L3HARRIS LDOS EBAY MELI SHOP SE PDD JD VIPS
    UPST RBLX SNOW CRWD ZM ROKU PINS SNAP TERM SPCE
"""

def sanitize_name(ticker: str) -> str:
    name = re.sub(r"[^A-Za-z0-9]+", "_", ticker.replace("^", "IDX_"))
    return name.strip("_")


def build_yf_dict() -> Dict[str, str]:
    tickers = []

    for t in YF_TICKERS_RAW.split():
        t = t.strip()
        if not t:
            continue
        if t in BAD_TICKERS:
            continue
        tickers.append(t)

    seen = set()
    unique_tickers = []

    for t in tickers:
        if t not in seen:
            seen.add(t)
            unique_tickers.append(t)

    yf_dict = {}
    used_names = set()

    for ticker in unique_tickers:
        base_name = MANUAL_YF_NAMES.get(ticker, sanitize_name(ticker))
        name = base_name
        i = 2

        while name in used_names:
            name = f"{base_name}_{i}"
            i += 1

        used_names.add(name)
        yf_dict[ticker] = name

    return yf_dict


# =============================================================================
# FRED INDICATORS
# =============================================================================

fred_dict = {
    "VIXCLS": "VIX",
    "VIXDVOL": "VIX_DrawVol",
    "OILPRICE": "Oil_Price",
    "SP500": "SP500_Level",
    "WILL5000IND": "Wilshire5000",

    "DCOILWTICO": "WTI_Oil_FRED",
    "DCOILBRENTEU": "Brent_Oil_FRED",

    "DGS30": "US30Y_Rate",
    "DGS20": "US20Y_Rate",
    "DGS10": "US10Y_Rate",
    "DGS7": "US7Y_Rate",
    "DGS5": "US5Y_Rate",
    "DGS3": "US3Y_Rate",
    "DGS2": "US2Y_Rate",
    "DGS1": "US1Y_Rate",
    "DTB6": "US6M_Rate",
    "DTB3": "US3M_Rate",
    "DTB1": "US1M_Rate",

    "FEDFUNDS": "FedFunds",
    "EFFR": "EFFR",
    "SOFR": "SOFR_SecuredOIS",
    "DFF": "DFF",

    "T10Y2Y": "T10Y2Y_Spread",
    "T10Y3M": "T10Y3M_Spread",
    "T10YIE": "T10Y_Inflation_Expectation",
    "T5YIE": "T5Y_Inflation_Expectation",
    "T5YIFR": "T5Y5Y_Inflation_Forward",
    "TEDRATE": "TED_Spread",
    "BAMLH0A0HYM2": "HY_OAS",
    "BAMLC0A0CM": "IG_OAS",
    "BAMLC0A4CBBB": "BBB_OAS",

    "UNRATE": "Unemployment",
    "PAYEMS": "NonfarmPayrolls",
    "CPIAUCSL": "CPI",
    "CPILFESL": "Core_CPI",
    "PCE": "PCE",
    "PCEPILFE": "Core_PCE",
    "GDP": "GDP",
    "INDPRO": "Industrial_Production",
    "UMCSENT": "Michigan_Sentiment",
    "RSAFS": "Retail_Sales",

    "NFCI": "NFCI",
    "STLFSI4": "STLFSI4",
}

In [10]:
def safe_series(df: pd.DataFrame, col: str) -> pd.Series:

    x = df.loc[:, col]

    # si plusieurs colonnes (cas MultiIndex / duplicates)
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, 0]

    # conversion safe
    x = pd.to_numeric(x, errors="coerce")

    # garantit alignement index
    x.index = df.index

    return x


def safe_bool_series(x: pd.Series) -> pd.Series:
    """
    Force une Series en bool propre.
    Évite le bug ~True = -2 / ~False = -1 si dtype=object.
    """
    return x.fillna(False).astype(bool)


In [11]:
# =============================================================================
# DATA LOADER
# =============================================================================

class DataLoader:
    def __init__(self):
        self.yf_failed = []
        self.fred_failed = []

    def load_yfinance_massive(self, tickers: Dict[str, str], start_date: str, end_date: str) -> pd.DataFrame:
        all_tickers = list(tickers.keys())
        chunks = [
            all_tickers[i:i + YF_CHUNK_SIZE]
            for i in range(0, len(all_tickers), YF_CHUNK_SIZE)
        ]

        all_data = []

        for idx, chunk in enumerate(chunks, 1):
            print(f"[DATA] Yahoo chunk {idx}/{len(chunks)} | tickers={len(chunk)}")

            try:
                raw = yf.download(
                    tickers=chunk,
                    start=start_date,
                    end=end_date,
                    progress=False,
                    auto_adjust=False,
                    group_by="ticker",
                    threads=True
                )

                if raw is None or raw.empty:
                    self.yf_failed.extend(chunk)
                    continue

                for ticker in chunk:
                    try:
                        col_name = tickers[ticker]

                        if isinstance(raw.columns, pd.MultiIndex):
                            if ticker not in raw.columns.get_level_values(0):
                                self.yf_failed.append(ticker)
                                continue

                            ticker_df = raw[ticker]

                            if "Close" not in ticker_df.columns:
                                self.yf_failed.append(ticker)
                                continue

                            close = ticker_df["Close"]

                        else:
                            if len(chunk) != 1 or "Close" not in raw.columns:
                                self.yf_failed.append(ticker)
                                continue

                            close = raw["Close"]

                        close = pd.to_numeric(close, errors="coerce")
                        close = close.rename(col_name).to_frame()
                        close.index = pd.to_datetime(close.index)
                        close = close[~close.index.duplicated(keep="last")]
                        close = close.sort_index()

                        if close.dropna().shape[0] >= 100:
                            all_data.append(close)
                        else:
                            self.yf_failed.append(ticker)

                    except Exception:
                        self.yf_failed.append(ticker)

            except Exception:
                self.yf_failed.extend(chunk)

            time.sleep(SLEEP_BETWEEN_CHUNKS)

        if not all_data:
            return pd.DataFrame()

        df = pd.concat(all_data, axis=1, join="outer").sort_index()
        df = df.loc[:, ~df.columns.duplicated()]
        return df

    def load_fred(self, indicators: Dict[str, str], start_date: str, end_date: str) -> pd.DataFrame:
        data = []

        for i, (code, col_name) in enumerate(indicators.items(), 1):
            print(f"[DATA] FRED {i}/{len(indicators)} | {code}")

            try:
                raw = pdr.get_data_fred(code, start=start_date, end=end_date)

                if raw is None or raw.empty:
                    self.fred_failed.append(code)
                    continue

                series = raw.iloc[:, 0]
                series = pd.to_numeric(series, errors="coerce")
                series = series.rename(col_name).to_frame()
                series.index = pd.to_datetime(series.index)
                series = series[~series.index.duplicated(keep="last")]
                series = series.sort_index()

                if series.dropna().shape[0] >= 30:
                    data.append(series)
                else:
                    self.fred_failed.append(code)

            except Exception:
                self.fred_failed.append(code)

            time.sleep(0.1)

        if not data:
            return pd.DataFrame()

        df = pd.concat(data, axis=1, join="outer").sort_index()
        df = df.loc[:, ~df.columns.duplicated()]
        return df

    def combine_to_latest_full_dataset(self, yf_df: pd.DataFrame, fred_df: pd.DataFrame) -> pd.DataFrame:
        """
        CORRECTIF (bug identifié) : l'ancienne version calculait la couverture
        de chaque colonne (notna().mean()) puis filtrait les LIGNES à >=95% de
        couverture. Comme plusieurs colonnes (tickers/indices créés après 2000,
        ex: OVX 2007, GVZ 2008) n'ont pas d'historique avant ~2010, la
        couverture moyenne des lignes < 2010 tombait sous le seuil et TOUTES
        les lignes pré-2010 étaient supprimées - peu importe MIN_DATA_START.
        Conséquence concrète : tout le sweep TRAIN_START=2000..2010 utilisait
        en réalité toujours la même fenêtre (~2010+), donc les 22 runs du
        sweep étaient quasi identiques entre eux.

        CORRECTIF appliqué : on calcule la couverture PAR COLONNE sur toute la
        fenêtre demandée (depuis MIN_DATA_START), et on retire en amont les
        colonnes dont la couverture est insuffisante - PAS les lignes. Une
        colonne sans historique avant 2010 est donc exclue du feature set
        pour ce run, mais les lignes 2000-2009 sont conservées avec les
        features qui, elles, couvrent bien toute la période.
        """
        if yf_df.empty:
            raise ValueError("Yahoo Finance data is empty.")

        yf_df = yf_df.sort_index()
        yf_df.index = pd.to_datetime(yf_df.index)
        yf_df = yf_df.loc[yf_df.index >= MIN_DATA_START].copy()
        yf_df = yf_df.dropna(axis=1, how="all")

        if yf_df.empty:
            raise ValueError("Yahoo Finance dataframe has no usable columns since MIN_DATA_START.")

        if not fred_df.empty:
            fred_df = fred_df.sort_index()
            fred_df.index = pd.to_datetime(fred_df.index)
            fred_df = fred_df.loc[fred_df.index >= MIN_DATA_START].copy()
            fred_df = fred_df.dropna(axis=1, how="all")

            fred_on_market_calendar = fred_df.reindex(yf_df.index).ffill()
            combined = pd.concat([yf_df, fred_on_market_calendar], axis=1)
        else:
            combined = yf_df.copy()

        combined = combined.loc[:, ~combined.columns.duplicated()]
        combined = combined.replace([np.inf, -np.inf], np.nan)
        combined = combined.sort_index()

        if "VIX_Price" not in combined.columns and "VIX" not in combined.columns:
            raise ValueError("No usable VIX column found after combining data.")

        # --- Filtre 1 : colonne apparue trop tard après MIN_DATA_START ---
        latest_allowed_first_valid = MIN_DATA_START + pd.Timedelta(days=MAX_FIRST_VALID_LAG_DAYS)

        keep_cols_start = []
        for col in combined.columns:
            first_valid = combined[col].first_valid_index()
            if first_valid is None:
                continue
            if first_valid <= latest_allowed_first_valid:
                keep_cols_start.append(col)

        combined = combined[keep_cols_start]

        if combined.empty:
            raise ValueError("No columns left after first-valid-date filter.")

        combined = combined.ffill()

        # --- Filtre 2 (CORRIGÉ) : couverture par COLONNE sur toute la fenêtre,
        # pas de filtre de ligne. Seuil MIN_COLUMN_COVERAGE (ex: 0.90). ---
        coverage = combined.notna().mean()
        keep_cols_coverage = coverage[coverage >= MIN_COLUMN_COVERAGE].index.tolist()

        core_cols = [
            "VIX_Price",
            "VIX",
            "SP500_Price",
            "SPY",
            "QQQ",
            "TLT_LongBond",
            "GLD_Gold",
            "USO_Oil"
        ]

        for c in core_cols:
            if c in combined.columns and c not in keep_cols_coverage:
                if combined[c].notna().mean() >= 0.75:
                    keep_cols_coverage.append(c)

        n_cols_before = combined.shape[1]
        combined = combined[keep_cols_coverage]
        print(f"[COMBINE] Colonnes retirées par couverture insuffisante (<{MIN_COLUMN_COVERAGE:.0%}): "
              f"{n_cols_before - combined.shape[1]}/{n_cols_before}")

        if combined.empty:
            raise ValueError("No columns left after coverage filter.")

        # --- PAS de filtre de ligne (row_coverage) ici : on garde toutes les
        # lignes depuis MIN_DATA_START, puisque les colonnes retenues couvrent
        # déjà >= MIN_COLUMN_COVERAGE de cette fenêtre par construction. ---
        combined = combined.ffill()
        combined = combined.dropna(axis=0, how="any")

        if combined.empty:
            raise ValueError("Final dataset is empty after coverage filters.")

        print(f"[COMBINE] Columns kept: {combined.shape[1]}")
        print(f"[COMBINE] Rows kept:    {combined.shape[0]}")
        print(f"[COMBINE] Date range:   {combined.index.min().date()} → {combined.index.max().date()}")

        return combined


In [12]:
class FeatureEngineer:
    def __init__(self):
        self.features = []

    def add(self, name: str):
        if name not in self.features:
            self.features.append(name)

    def create_features(self, df: pd.DataFrame):
        print("[FEATURES] Creating features...")

        df = df.copy()
        df = df.loc[:, ~df.columns.duplicated()]
        base_cols = list(df.columns)

        for col in base_cols:
            s = safe_series(df, col)
            # Replace 0 values with NaN to avoid ZeroDivisionError in pct_change
            s_clean = s.replace(0, np.nan)

            # returns
            for p in [1, 5, 20]:
                f = f"{col}_ret_{p}d"
                df[f] = s_clean.pct_change(p)
                self.add(f)

            # volatility
            f = f"{col}_vol_20d"
            df[f] = s_clean.pct_change().rolling(20).std()
            self.add(f)

            # z-score
            mean = s_clean.rolling(60).mean()
            std = s_clean.rolling(60).std()
            f = f"{col}_zscore_60d"
            df[f] = (s_clean - mean) / (std + 1e-8)
            self.add(f)

        # VIX features
        vix_col = "VIX_Price" if "VIX_Price" in df.columns else "VIX" if "VIX" in df.columns else None

        if vix_col:
            vix = safe_series(df, vix_col)

            df["vix_level"] = vix
            df["vix_change_1d"] = vix.pct_change(1)
            df["vix_change_5d"] = vix.pct_change(5)
            df["vix_ma_20"] = vix.rolling(20).mean()
            df["vix_vs_ma20"] = vix - df["vix_ma_20"]

            for f in [
                "vix_level",
                "vix_change_1d",
                "vix_change_5d",
                "vix_ma_20",
                "vix_vs_ma20"
            ]:
                self.add(f)

        # SP500 features
        if "SP500_Price" in df.columns:
            spx = safe_series(df, "SP500_Price")

            df["spx_realized_vol_20d"] = spx.pct_change().rolling(20).std() * np.sqrt(252)

            # Fix for spx_drawdown_252d to prevent data leakage:
            # Calculate the peak from the *previous* 252 days, excluding the current day.
            # This ensures the feature for day 't' only uses data available up to day 't-1'.
            spx_peak_before_today = spx.rolling(window=252, closed='left').max()
            df["spx_drawdown_252d"] = (spx_peak_before_today - spx) / (spx_peak_before_today + 1e-8)
            df["spx_drawdown_252d"] = df["spx_drawdown_252d"].clip(lower=0) # Drawdown cannot be negative

            df["spx_down_day"] = (spx.pct_change(1) < 0).astype(int)

            for f in [
                "spx_realized_vol_20d",
                "spx_drawdown_252d",
                "spx_down_day"
            ]:
                self.add(f)

        # Yield curve
        if "US10Y_Rate" in df.columns and "US2Y_Rate" in df.columns:
            us10 = safe_series(df, "US10Y_Rate")
            us2 = safe_series(df, "US2Y_Rate")

            df["yield_curve_10y_2y"] = us10 - us2
            df["yield_curve_change_20d"] = df["yield_curve_10y_2y"].diff(20)

            for f in [
                "yield_curve_10y_2y",
                "yield_curve_change_20d"
            ]:
                self.add(f)

        self.features = [f for f in self.features if f in df.columns]

        return df, self.features

In [13]:
class TargetBuilder:
    """
    Construit la cible (VIX_Direction) et le régime de marché (VIX_Regime).

    horizon_days controle l'horizon de prediction : la cible devient
    "le VIX monte-t-il entre T et T+horizon_days" au lieu de T et T+1 fixe.
    Avec horizon_days=1 (defaut), comportement strictement identique a avant.

    Deux modes pour définir les seuils de régime CALM/NORMAL/STRESS :

    - mode="fixed"   : quantiles q33/q67 calculés UNE FOIS sur la période
                        train (< train_end), puis appliqués tels quels à
                        tout le dataframe (train + test). Pas de fuite
                        train->test, mais le seuil ne s'adapte pas si le
                        régime de volatilité change structurellement avec
                        le temps (ex: VIX 2008 vs VIX 2017).

    - mode="rolling" : quantiles q33/q67 recalculés à CHAQUE date T sur une
                        fenêtre glissante des `rolling_window` jours
                        précédents (closed='left', donc strictement avant T
                        - pas de fuite intra-jour). Le régime à la date T
                        reflète le niveau de VIX relatif à son contexte
                        récent (~2 ans avec rolling_window=504), pas à toute
                        l'histoire 2000-2026 mélangée.
    """
    def __init__(self, q_low: float = 0.33, q_high: float = 0.67,
                 mode: str = "fixed", rolling_window: int = 504,
                 horizon_days: int = 1):
        assert mode in ("fixed", "rolling"), "mode must be 'fixed' or 'rolling'"
        assert horizon_days >= 1, "horizon_days must be >= 1"
        self.vix_col = None
        self.q_low = q_low
        self.q_high = q_high
        self.mode = mode
        self.rolling_window = rolling_window
        self.horizon_days = horizon_days  # horizon de prediction en jours de bourse (1, 3, 5...)
        self.calm_threshold_ = None    # scalar if mode="fixed", else None
        self.stress_threshold_ = None
        self.flat_threshold = VIX_FLAT_PCT_THRESHOLD
        self.flat_removed = 0
        self.total_before_flat_filter = 0

    def build(self, df: pd.DataFrame, train_end: str = None) -> pd.DataFrame:
        df = df.copy()
        df = df.loc[:, ~df.columns.duplicated()]

        if "VIX_Price" in df.columns:
            self.vix_col = "VIX_Price"
        elif "VIX" in df.columns:
            self.vix_col = "VIX"
        else:
            raise KeyError("No VIX column found.")

        vix = safe_series(df, self.vix_col)
        # Horizon de prediction : shift(-horizon_days) au lieu de shift(-1) fixe.
        # horizon_days=1 reproduit exactement le comportement d'origine.
        future_vix = vix.shift(-self.horizon_days)

        vix_next_change = (future_vix / vix) - 1

        df["VIX_Next_Change"] = vix_next_change
        df["VIX_Is_Flat"] = vix_next_change.abs() <= self.flat_threshold

        direction = pd.Series(np.nan, index=df.index)
        direction.loc[vix_next_change > 0] = 1
        direction.loc[vix_next_change <= 0] = 0
        direction.loc[future_vix.isna()] = np.nan

        df["VIX_Direction"] = direction
        df.loc[future_vix.isna(), "VIX_Is_Flat"] = np.nan

        if self.mode == "fixed":
            # --- Quantiles fixes, calculés sur le train seulement (no leakage) ---
            if train_end is not None:
                vix_for_quantiles = vix.loc[vix.index < pd.Timestamp(train_end)]
            else:
                vix_for_quantiles = vix

            self.calm_threshold_ = vix_for_quantiles.quantile(self.q_low)
            self.stress_threshold_ = vix_for_quantiles.quantile(self.q_high)

            regime = pd.Series("NORMAL", index=df.index)
            regime.loc[vix < self.calm_threshold_] = "CALM"
            regime.loc[vix >= self.stress_threshold_] = "STRESS"

            threshold_desc = (
                f"CALM < {self.calm_threshold_:.2f}, "
                f"NORMAL [{self.calm_threshold_:.2f}-{self.stress_threshold_:.2f}), "
                f"STRESS >= {self.stress_threshold_:.2f}  (fixe, calculé sur train)"
            )

        else:
            # --- Quantiles rolling : recalculés à chaque date T sur les
            # `rolling_window` jours STRICTEMENT précédents (closed='left').
            # Pas de fuite : le quantile à T n'utilise jamais VIX(T) ni le futur.
            rolling_calm = vix.rolling(window=self.rolling_window, closed='left').quantile(self.q_low)
            rolling_stress = vix.rolling(window=self.rolling_window, closed='left').quantile(self.q_high)

            self.calm_threshold_ = rolling_calm   # Series, pas un scalaire
            self.stress_threshold_ = rolling_stress

            regime = pd.Series("NORMAL", index=df.index)
            regime.loc[vix < rolling_calm] = "CALM"
            regime.loc[vix >= rolling_stress] = "STRESS"
            # Tant que la fenêtre rolling n'est pas pleine (début d'historique),
            # rolling_calm/rolling_stress sont NaN -> régime indéfini -> ces
            # lignes seront retirées plus bas (dropna sur VIX_Regime_valid).
            regime.loc[rolling_calm.isna() | rolling_stress.isna()] = np.nan

            threshold_desc = (
                f"rolling sur {self.rolling_window} jours (~{self.rolling_window/252:.1f} ans), "
                f"recalculé à chaque date T sur les jours STRICTEMENT antérieurs à T"
            )

        df["VIX_Regime"] = regime

        # Lignes à retirer : direction NaN (fin de série), OU régime NaN (mode
        # rolling, début de série sans assez d'historique pour la fenêtre)
        df = df.dropna(subset=["VIX_Direction", "VIX_Is_Flat", "VIX_Regime"]).copy()

        df["VIX_Is_Flat"] = df["VIX_Is_Flat"].fillna(False).astype(bool)

        self.total_before_flat_filter = int(len(df))
        self.flat_removed = int(df["VIX_Is_Flat"].sum())

        df = df.loc[~df["VIX_Is_Flat"]].copy()

        df["VIX_Direction"] = df["VIX_Direction"].astype(int)
        df["VIX_Is_Flat"] = df["VIX_Is_Flat"].astype(bool)

        print(f"[TARGET] VIX column: {self.vix_col}  |  mode={self.mode}")
        print(f"[TARGET] VIX regime thresholds: {threshold_desc}")
        print(f"[TARGET] VIX flat threshold: ±{self.flat_threshold:.2%}")
        print(f"[TARGET] Flat days removed completely: {self.flat_removed}/{self.total_before_flat_filter}")
        print(f"[TARGET] Remaining non-flat rows: {len(df)}")
        print(df["VIX_Regime"].value_counts())

        return df



In [14]:
class TrainFittedCleaner:
    def __init__(self):
        self.medians = None
        self.lower = None
        self.upper = None

    def fit(self, X: pd.DataFrame):
        X = X.replace([np.inf, -np.inf], np.nan)
        self.medians = X.median()
        self.lower = X.quantile(0.01)
        self.upper = X.quantile(0.99)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        X = X.replace([np.inf, -np.inf], np.nan)
        X = X.fillna(self.medians)
        X = X.clip(lower=self.lower, upper=self.upper, axis=1)
        X = X.fillna(0)
        return X

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        self.fit(X)
        return self.transform(X)

In [15]:
def evaluate_model_across_regimes(df_test, trained_by_regime):
    all_preds = pd.Series(index=df_test.index, dtype=float)
    all_probas = pd.Series(index=df_test.index, dtype=float)

    for regime, pack in trained_by_regime.items():
        mask = df_test["VIX_Regime"] == regime

        if mask.sum() == 0:
            continue

        X_raw = df_test.loc[mask, pack["features"]]
        X_clean = pack["cleaner"].transform(X_raw)
        X_scaled = pack["scaler"].transform(X_clean)

        model = pack["model"]

        pred = model.predict(X_scaled)

        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_scaled)[:, 1]
        else:
            proba = pred.astype(float)

        all_preds.loc[mask] = pred
        all_probas.loc[mask] = proba

    valid = all_preds.notna()

    y_true = df_test.loc[valid, "VIX_Direction"].values
    y_pred = all_preds.loc[valid].astype(int).values
    y_proba = all_probas.loc[valid].values

    if len(y_true) == 0:
        return None

    return compute_metrics(y_true, y_pred, y_proba)

In [16]:
def model_configs():
    return {
        "XGBoost": (
            XGBClassifier,
            {
                "max_depth": [2, 3],
                "learning_rate": [0.03, 0.05],
                "n_estimators": [75, 125],
                "subsample": [0.8],
                "colsample_bytree": [0.8]
            },
            {
                "random_state": RANDOM_STATE,
                "eval_metric": "logloss",
                "n_jobs": -1
            }
        ),
        "LightGBM": (
            LGBMClassifier,
            {
                "num_leaves": [7, 15],
                "learning_rate": [0.03, 0.05],
                "n_estimators": [75, 125],
                "max_depth": [3, 5]
            },
            {
                "random_state": RANDOM_STATE,
                "verbose": -1,
                "class_weight": "balanced"
            }
        ),
        "GradientBoosting": (
            GradientBoostingClassifier,
            {
                "n_estimators": [75, 125],
                "learning_rate": [0.03, 0.05],
                "max_depth": [2, 3],
                "min_samples_leaf": [10]
            },
            {
                "random_state": RANDOM_STATE
            }
        ),
        "RandomForest": (
            RandomForestClassifier,
            {
                "n_estimators": [150],
                "max_depth": [3, 5],
                "min_samples_leaf": [10, 20]
            },
            {
                "random_state": RANDOM_STATE,
                "n_jobs": -1,
                "class_weight": "balanced"
            }
        ),
        "LogisticRegression": (
            LogisticRegression,
            {
                "C": [0.01, 0.1, 1.0],
                "penalty": ["l2"]
            },
            {
                "random_state": RANDOM_STATE,
                "max_iter": 2000,
                "class_weight": "balanced"
            }
        ),
    }


In [17]:
def compute_metrics(y_true, y_pred, y_proba):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "R2": r2_score(y_true, y_proba),
        "AUC": roc_auc_score(y_true, y_proba) if len(np.unique(y_true)) > 1 else np.nan,
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "Pred_0": int((y_pred == 0).sum()),
        "Pred_1": int((y_pred == 1).sum()),
        "Actual_0": int((y_true == 0).sum()),
        "Actual_1": int((y_true == 1).sum()),
        "Confusion_Matrix": cm.tolist()
    }

In [18]:
print("[INIT] Building ticker dictionaries...")
yf_dict = build_yf_dict()
print(f"[INIT] Yahoo tickers: {len(yf_dict)}")
print(f"[INIT] FRED indicators: {len(fred_dict)}")

[INIT] Building ticker dictionaries...
[INIT] Yahoo tickers: 345
[INIT] FRED indicators: 43


In [19]:
loader = DataLoader()
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")
print("[STEP 1/6] Loading data...")
yf_df = loader.load_yfinance_massive(yf_dict, TRAIN_START, end_date)
fred_df = loader.load_fred(fred_dict, TRAIN_START, end_date)

print("[STEP 2/6] Combining data...")
df = loader.combine_to_latest_full_dataset(yf_df, fred_df)


[STEP 1/6] Loading data...
[DATA] Yahoo chunk 1/9 | tickers=40
[DATA] Yahoo chunk 2/9 | tickers=40
[DATA] Yahoo chunk 3/9 | tickers=40
[DATA] Yahoo chunk 4/9 | tickers=40


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['TBP']: YFPricesMissingError('possibly delisted; no price data found  (1d 2000-01-01 -> 2026-07-07)')


[DATA] Yahoo chunk 5/9 | tickers=40


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: LVRK"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['LVRK']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] Yahoo chunk 6/9 | tickers=40


...

[DATA] Yahoo chunk 7/9 | tickers=40


ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['CYB', 'BZF']: YFPricesMissingError('possibly delisted; no price data found  (1d 2000-01-01 -> 2026-07-07)')


[DATA] Yahoo chunk 8/9 | tickers=40
[DATA] Yahoo chunk 9/9 | tickers=25


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['L3HARRIS']: YFTzMissingError('possibly delisted; no timezone found')


[DATA] FRED 1/43 | VIXCLS
[DATA] FRED 2/43 | VIXDVOL
[DATA] FRED 3/43 | OILPRICE
[DATA] FRED 4/43 | SP500
[DATA] FRED 5/43 | WILL5000IND
[DATA] FRED 6/43 | DCOILWTICO
[DATA] FRED 7/43 | DCOILBRENTEU
[DATA] FRED 8/43 | DGS30
[DATA] FRED 9/43 | DGS20
[DATA] FRED 10/43 | DGS10
[DATA] FRED 11/43 | DGS7
[DATA] FRED 12/43 | DGS5
[DATA] FRED 13/43 | DGS3
[DATA] FRED 14/43 | DGS2
[DATA] FRED 15/43 | DGS1
[DATA] FRED 16/43 | DTB6
[DATA] FRED 17/43 | DTB3
[DATA] FRED 18/43 | DTB1
[DATA] FRED 19/43 | FEDFUNDS
[DATA] FRED 20/43 | EFFR
[DATA] FRED 21/43 | SOFR
[DATA] FRED 22/43 | DFF
[DATA] FRED 23/43 | T10Y2Y
[DATA] FRED 24/43 | T10Y3M
[DATA] FRED 25/43 | T10YIE
[DATA] FRED 26/43 | T5YIE
[DATA] FRED 27/43 | T5YIFR
[DATA] FRED 28/43 | TEDRATE
[DATA] FRED 29/43 | BAMLH0A0HYM2
[DATA] FRED 30/43 | BAMLC0A0CM
[DATA] FRED 31/43 | BAMLC0A4CBBB
[DATA] FRED 32/43 | UNRATE
[DATA] FRED 33/43 | PAYEMS
[DATA] FRED 34/43 | CPIAUCSL
[DATA] FRED 35/43 | CPILFESL
[DATA] FRED 36/43 | PCE
[DATA] FRED 37/43 | PCEPILF

In [20]:
print("[STEP 3/6] Feature engineering...")
engineer = FeatureEngineer()
df, features = engineer.create_features(df)

[STEP 3/6] Feature engineering...
[FEATURES] Creating features...


In [21]:
# Copie de référence du dataframe après feature engineering, AVANT target/régime.
# Sert de point de départ identique pour les deux modes de quantile (fixed/rolling).
df_post_features = df.copy()
features_post_engineering = list(features)
print(f"[CHECKPOINT] df_post_features: {df_post_features.shape}, features: {len(features_post_engineering)}")


[CHECKPOINT] df_post_features: (6738, 1180), features: 985


In [22]:
print("[STEP] Chargement et feature engineering terminés. "
      "Démarrage de la Phase 1 (scoring des fenêtres par régime).")


[STEP] Chargement et feature engineering terminés. Démarrage de la Phase 1 (scoring des fenêtres par régime).


In [23]:
# =============================================================================
# FEATURES D'AMPLITUDE — calculées directement sur VIX et SPX
# Objectif : capturer l'ACCÉLÉRATION et l'ERRATICITÉ du VIX, pas seulement
# la direction. Ce sont les features manquantes pour prédire UP_FORT.
# =============================================================================

def add_amplitude_features(df: pd.DataFrame) -> pd.DataFrame:
    """Ajoute des features spécifiques à l'intensité du mouvement VIX."""
    df = df.copy()

    # Colonne VIX brute
    vix_col = "VIX_Price" if "VIX_Price" in df.columns else "VIX"
    spx_col = "SP500_Price" if "SP500_Price" in df.columns else None
    vix = df[vix_col]

    # 1. Volatilité-de-la-volatilité : vol réalisée du VIX sur 5 et 10 jours
    df["vix_vol_of_vol_5d"]  = vix.pct_change().rolling(5,  min_periods=3).std()
    df["vix_vol_of_vol_10d"] = vix.pct_change().rolling(10, min_periods=5).std()

    # 2. Momentum à court terme : accélération récente du VIX
    vix_ret1 = vix.pct_change(1)
    df["vix_momentum_2d"] = vix.pct_change(2)
    df["vix_momentum_3d"] = vix.pct_change(3)

    # 3. Accélération : changement de vitesse (dérivée seconde)
    df["vix_acceleration_1d"] = vix_ret1 - vix_ret1.shift(1)
    df["vix_acceleration_3d"] = vix_ret1 - vix_ret1.shift(3)

    # 4. Sur-extension court terme : distance au MA5 et MA10
    ma5  = vix.rolling(5,  min_periods=3).mean()
    ma10 = vix.rolling(10, min_periods=5).mean()
    df["vix_vs_ma5"]  = (vix - ma5)  / ma5.replace(0, np.nan)
    df["vix_vs_ma10"] = (vix - ma10) / ma10.replace(0, np.nan)

    # 5. Z-score court terme du VIX (5 et 10 jours)
    df["vix_zscore_5d"]  = (vix - ma5)  / vix.rolling(5,  min_periods=3).std().replace(0, np.nan)
    df["vix_zscore_10d"] = (vix - ma10) / vix.rolling(10, min_periods=5).std().replace(0, np.nan)

    # 6. Persistance du stress : % des 10 derniers jours avec VIX > MA20
    ma20 = vix.rolling(20, min_periods=10).mean()
    above_ma20 = (vix > ma20).astype(float)
    df["vix_pct_above_ma20_10d"] = above_ma20.rolling(10, min_periods=5).mean()

    # 7. Erraticité : max des |rendements| sur 5 jours vs moyenne
    abs_ret = vix.pct_change().abs()
    df["vix_max_abs_ret_5d"]  = abs_ret.rolling(5, min_periods=3).max()
    df["vix_mean_abs_ret_5d"] = abs_ret.rolling(5, min_periods=3).mean()
    df["vix_erratic_ratio"]   = df["vix_max_abs_ret_5d"] / df["vix_mean_abs_ret_5d"].replace(0, np.nan)

    # 8. Ratio vol_5d / vol_60d (spike court terme vs bruit de fond)
    df["vix_vol_ratio_5_60"] = (
        vix.pct_change().rolling(5,  min_periods=3).std() /
        vix.pct_change().rolling(60, min_periods=30).std().replace(0, np.nan)
    )

    # 9. SPX amplitude features (si disponible)
    if spx_col and spx_col in df.columns:
        spx = df[spx_col]
        spx_ret = spx.pct_change()
        df["spx_vol_5d"]         = spx_ret.rolling(5,  min_periods=3).std()
        df["spx_momentum_3d"]    = spx.pct_change(3)
        df["spx_abs_ret_max_5d"] = spx_ret.abs().rolling(5, min_periods=3).max()

    df = df.replace([np.inf, -np.inf], np.nan)
    n_added = sum(1 for c in df.columns if c in [
        "vix_vol_of_vol_5d","vix_vol_of_vol_10d","vix_momentum_2d","vix_momentum_3d",
        "vix_acceleration_1d","vix_acceleration_3d","vix_vs_ma5","vix_vs_ma10",
        "vix_zscore_5d","vix_zscore_10d","vix_pct_above_ma20_10d",
        "vix_max_abs_ret_5d","vix_mean_abs_ret_5d","vix_erratic_ratio",
        "vix_vol_ratio_5_60","spx_vol_5d","spx_momentum_3d","spx_abs_ret_max_5d"
    ])
    print(f"[AMPLITUDE FEATURES] {n_added} features d'amplitude ajoutées")
    return df

df_post_features = add_amplitude_features(df_post_features)

# Mettre à jour la liste de features
amplitude_feat_names = [
    "vix_vol_of_vol_5d","vix_vol_of_vol_10d","vix_momentum_2d","vix_momentum_3d",
    "vix_acceleration_1d","vix_acceleration_3d","vix_vs_ma5","vix_vs_ma10",
    "vix_zscore_5d","vix_zscore_10d","vix_pct_above_ma20_10d",
    "vix_max_abs_ret_5d","vix_mean_abs_ret_5d","vix_erratic_ratio",
    "vix_vol_ratio_5_60","spx_vol_5d","spx_momentum_3d","spx_abs_ret_max_5d"
]
new_feats = [f for f in amplitude_feat_names if f in df_post_features.columns]
features_post_engineering = features_post_engineering + new_feats
print(f"[AMPLITUDE FEATURES] features_post_engineering : {len(features_post_engineering)} total ({len(new_feats)} nouvelles)")


[AMPLITUDE FEATURES] 18 features d'amplitude ajoutées
[AMPLITUDE FEATURES] features_post_engineering : 1003 total (18 nouvelles)


In [24]:
# =============================================================================
# SPLIT 80/20 CHRONOLOGIQUE + EGARCH SPX → FEATURE VARIANCE CONDITIONNELLE
# =============================================================================

# Définition du split 80/20
all_dates = df_post_features.dropna(how="all").index.sort_values()
split_idx = int(len(all_dates) * 0.80)
TEST_DATE  = all_dates[split_idx].strftime("%Y-%m-%d")
print(f"[SPLIT 80/20] Train : {all_dates[0].date()} → {all_dates[split_idx-1].date()} ({split_idx} jours)")
print(f"[SPLIT 80/20] Test  : {all_dates[split_idx].date()} → {all_dates[-1].date()} ({len(all_dates)-split_idx} jours)")

# SPX log-rendements
spx_col = "SP500_Price" if "SP500_Price" in df_post_features.columns else None
if spx_col is None:
    raise ValueError("SP500_Price introuvable dans df_post_features")

spx     = df_post_features[spx_col].dropna().sort_index()
spx_ret = np.log(spx / spx.shift(1)).dropna() * 100  # ×100 pour stabilité numérique

spx_train = spx_ret.loc[spx_ret.index < pd.Timestamp(TEST_DATE)]
spx_test  = spx_ret.loc[spx_ret.index >= pd.Timestamp(TEST_DATE)]

print(f"\n[EGARCH SPX] Fit initial sur train ({len(spx_train)} jours)...")
am_base = arch_model(spx_train, vol="EGARCH", p=1, q=1, dist="skewt", rescale=False)
res_base = am_base.fit(disp="off", show_warning=False)
print(f"  AIC={res_base.aic:.1f}  BIC={res_base.bic:.1f}")
print(f"  Params : {dict(res_base.params.round(4))}")

# Variance conditionnelle EGARCH sur le train (1-step fitted values)
spx_train_cond_var = pd.Series(
    res_base.conditional_volatility**2 / 10000,
    index=spx_train.index,
    name="EGARCH_SPX_condvar_h1"
)

# Rolling forecast sur le test pour h=1,3,5
print(f"\n[EGARCH SPX] Rolling forecast sur test ({len(spx_test)} jours) — peut prendre plusieurs minutes...")

history = spx_train.copy()
egarch_preds = {h: [] for h in EGARCH_HORIZONS}
egarch_dates = []

for i, date in enumerate(spx_test.index):
    am  = arch_model(history, vol="EGARCH", p=1, q=1, dist="skewt", rescale=False)
    try:
        res = am.fit(disp="off", show_warning=False,
                     starting_values=res_base.params.values)
    except Exception:
        res = res_base  # fallback si convergence échoue

    for h in EGARCH_HORIZONS:
        if h == 1:
            fc     = res.forecast(horizon=1)
            sigma2 = fc.variance.iloc[-1, 0] / 10000
        else:
            try:
                sim    = res.forecast(horizon=h, method="simulation", simulations=300)
                sigma2 = sim.variance.iloc[-1, h-1] / 10000
            except Exception:
                sigma2 = np.nan
        egarch_preds[h].append(sigma2)

    egarch_dates.append(date)
    history.loc[date] = spx_test.loc[date]

    if (i+1) % 100 == 0:
        print(f"  {i+1}/{len(spx_test)} jours traités...")

# Construire les séries de variance conditionnelle SPX par horizon
for h in EGARCH_HORIZONS:
    col_name = f"EGARCH_SPX_condvar_h{h}"
    # train : valeurs fittées ; test : rolling forecast
    train_vals = spx_train_cond_var.rename(col_name)
    test_vals  = pd.Series(egarch_preds[h], index=egarch_dates, name=col_name)
    full_series = pd.concat([train_vals, test_vals]).sort_index()

    # Variation de variance (signal directionnel, meilleur que le niveau brut)
    full_series_delta = full_series.diff().rename(f"EGARCH_SPX_delta_h{h}")

    df_post_features[col_name] = full_series
    df_post_features[f"EGARCH_SPX_delta_h{h}"] = full_series_delta

egarch_feat_names = (
    [f"EGARCH_SPX_condvar_h{h}" for h in EGARCH_HORIZONS] +
    [f"EGARCH_SPX_delta_h{h}"   for h in EGARCH_HORIZONS]
)
new_egarch = [f for f in egarch_feat_names if f in df_post_features.columns]
features_post_engineering = features_post_engineering + new_egarch
print(f"\n[EGARCH SPX] {len(new_egarch)} features ajoutées (condvar + delta par horizon)")
print(f"[TOTAL FEATURES] {len(features_post_engineering)}")


[SPLIT 80/20] Train : 2000-08-11 → 2021-04-30 (5390 jours)
[SPLIT 80/20] Test  : 2021-05-03 → 2026-07-06 (1348 jours)

[EGARCH SPX] Fit initial sur train (5389 jours)...
  AIC=14242.5  BIC=14282.1
  Params : {'mu': np.float64(0.0566), 'omega': np.float64(0.0138), 'alpha[1]': np.float64(0.2399), 'beta[1]': np.float64(0.9845), 'eta': np.float64(5.4013), 'lambda': np.float64(-0.0794)}

[EGARCH SPX] Rolling forecast sur test (1348 jours) — peut prendre plusieurs minutes...
  100/1348 jours traités...
  200/1348 jours traités...
  300/1348 jours traités...
  400/1348 jours traités...
  500/1348 jours traités...
  600/1348 jours traités...
  700/1348 jours traités...
  800/1348 jours traités...
  900/1348 jours traités...
  1000/1348 jours traités...
  1100/1348 jours traités...
  1200/1348 jours traités...
  1300/1348 jours traités...

[EGARCH SPX] 6 features ajoutées (condvar + delta par horizon)
[TOTAL FEATURES] 1009


In [25]:
# Kalman + HMM (identique V2, mais TEST_DATE est maintenant dynamique)
vix_col = "VIX_Price" if "VIX_Price" in df_post_features.columns else "VIX"
vix = df_post_features[vix_col].dropna().sort_index()
vix_log_ret = np.log(vix / vix.shift(1))

# Kalman
vix_array     = vix.values.reshape(-1, 1)
vix_train_arr = vix.loc[vix.index < pd.Timestamp(TEST_DATE)].values.reshape(-1,1)
kf = KalmanFilter(transition_matrices=np.array([[1]]),
                  observation_matrices=np.array([[1]]),
                  initial_state_mean=np.array([vix.iloc[0]]),
                  initial_state_covariance=np.array([[1.0]]),
                  em_vars=["transition_covariance","observation_covariance"])
kf = kf.em(vix_train_arr, n_iter=20)
state_means, _ = kf.filter(vix_array)
state_smooth, _ = kf.smooth(vix_array)
vix_kalman = pd.Series(state_means[:,0], index=vix.index)
vix_smooth = pd.Series(state_smooth[:,0], index=vix.index)
vix_residual   = vix - vix_kalman
vix_innovation = vix - vix_smooth.shift(1)
print(f"[Kalman] Q={kf.transition_covariance[0,0]:.4f}  R={kf.observation_covariance[0,0]:.4f}")

# HMM
rv5d = vix_log_ret.pow(2).rolling(5, min_periods=3).mean()
mu_tr = vix.loc[vix.index < pd.Timestamp(TEST_DATE)].mean()
sd_tr = vix.loc[vix.index < pd.Timestamp(TEST_DATE)].std()
vix_norm = (vix - mu_tr) / sd_tr
X_full = pd.DataFrame({"ret": vix_log_ret, "vol5d": np.sqrt(rv5d), "level": vix_norm}).dropna()
X_tr_hmm = X_full.loc[X_full.index < pd.Timestamp(TEST_DATE)].values
model_hmm = hmmlib.GaussianHMM(n_components=2, covariance_type="full",
                                n_iter=200, random_state=RANDOM_STATE)
model_hmm.fit(X_tr_hmm)
states_tr = model_hmm.predict(X_tr_hmm)
state_vol = [np.sqrt(rv5d.reindex(X_full.loc[X_full.index < pd.Timestamp(TEST_DATE)].index)
             .values[states_tr == s]).mean() for s in range(2)]
stress_state = int(np.argmax(state_vol))
proba_full = model_hmm.predict_proba(X_full.values)
p_stress = pd.Series(proba_full[:, stress_state], index=X_full.index)

df_post_features["VIX_Residual"]   = vix_residual
df_post_features["VIX_Innovation"] = vix_innovation
df_post_features["P_stress_HMM"]   = p_stress

ts_feats = ["VIX_Residual","VIX_Innovation","P_stress_HMM"]
features_post_engineering = features_post_engineering + [f for f in ts_feats if f in df_post_features.columns]
print(f"[TS FEATURES] Kalman + HMM ajoutés → {len(features_post_engineering)} features total")


[Kalman] Q=2.0390  R=0.6358
[TS FEATURES] Kalman + HMM ajoutés → 1012 features total


In [26]:
# =============================================================================
# FEATURES MATHÉMATIQUES AVANCÉES
# 1. VRP  — Variance Risk Premium : VIX² - E[RV_30j] via HAR-RV
# 2. Jump Intensity — fréquence des sauts |r| > 3σ rolling 60j
# 3. Implied Correlation — proxy via secteurs ETF disponibles
# 4. Hawkes Process — intensité de clustering des pics VIX
# =============================================================================

vix_col = "VIX_Price" if "VIX_Price" in df_post_features.columns else "VIX"
spx_col = "SP500_Price" if "SP500_Price" in df_post_features.columns else None

vix = df_post_features[vix_col].dropna().sort_index()
vix_ret = vix.pct_change()

# ── 1. VRP (Variance Risk Premium) ──────────────────────────────────────────
# VRP = VIX² - E[RV_30j]  où E[RV] estimé par HAR-RV sur train
# Signal documenté : VRP > 0 → VIX tend à baisser (prime de risque élevée)
rv1d  = vix_ret.pow(2)
rv5d  = rv1d.rolling(5,  min_periods=3).mean()
rv22d = rv1d.rolling(22, min_periods=10).mean()

# HAR-RV : prédiction RV à 30j
rv_target = rv1d.rolling(30, min_periods=15).mean().shift(-30)
har_df = pd.DataFrame({"rv1":rv1d,"rv5":rv5d,"rv22":rv22d,"y":rv_target}).dropna()
tr_mask = har_df.index < pd.Timestamp(TEST_DATE)
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
X_har = sm.add_constant(har_df.loc[tr_mask, ["rv1","rv5","rv22"]])
y_har = har_df.loc[tr_mask, "y"]
har_model = sm.OLS(y_har, X_har).fit()
X_full_har = sm.add_constant(har_df[["rv1","rv5","rv22"]])
rv_forecast = har_model.predict(X_full_har)

vix_sq = (vix / 100).pow(2)   # VIX en décimal²
vrp = vix_sq.reindex(rv_forecast.index) - rv_forecast
vrp.name = "VRP"

# Dérivés du VRP
vrp_zscore = (vrp - vrp.loc[vrp.index < pd.Timestamp(TEST_DATE)].mean())            / vrp.loc[vrp.index < pd.Timestamp(TEST_DATE)].std()
vrp_zscore.name = "VRP_zscore"
vrp_ma5 = vrp.rolling(5, min_periods=3).mean()
vrp_ma5.name = "VRP_ma5"

print(f"[VRP] train mean={vrp.loc[vrp.index < pd.Timestamp(TEST_DATE)].mean():.6f}, "
      f"std={vrp.loc[vrp.index < pd.Timestamp(TEST_DATE)].std():.6f}")

# ── 2. Jump Intensity ────────────────────────────────────────────────────────
# λ_t = proportion de jours avec |r_t| > 3σ sur fenêtre rolling 60j
# Capte l'erraticité mieux que la vol réalisée simple
sigma_60 = vix_ret.rolling(60, min_periods=30).std()
is_jump   = (vix_ret.abs() > 3 * sigma_60).astype(float)
jump_intensity_60d = is_jump.rolling(60, min_periods=30).mean()
jump_intensity_60d.name = "jump_intensity_60d"

# Variante 20j (plus réactive)
jump_intensity_20d = is_jump.rolling(20, min_periods=10).mean()
jump_intensity_20d.name = "jump_intensity_20d"

print(f"[JUMP] train mean λ_60d={jump_intensity_60d.loc[jump_intensity_60d.index < pd.Timestamp(TEST_DATE)].mean():.4f}")

# ── 3. Implied Correlation (proxy) ──────────────────────────────────────────
# ρ_impl ≈ (VIX² - Σwᵢ²σᵢ²) / (Σᵢ≠ⱼ wᵢwⱼσᵢσⱼ)
# Proxy simplifié : VXN/VIX ratio — quand VXN > VIX, corrélations tech > marché global
# Signal : ratio élevé → stress idiosyncratique (pas systémique)
sector_pairs = [
    ("VXN_NASDAQ_Vol_Price","VIX_Price"),  # NASDAQ vs SPX vol
    ("OVX_Oil_Vol_Price","VIX_Price"),     # Oil vol vs SPX vol
]
for num_col, den_col in sector_pairs:
    if num_col in df_post_features.columns and den_col in df_post_features.columns:
        col_name = f"impl_corr_proxy_{num_col.split('_')[0]}"
        num = df_post_features[num_col]
        den = df_post_features[den_col].replace(0, np.nan)
        ratio = num / den
        ratio_train = ratio.loc[ratio.index < pd.Timestamp(TEST_DATE)]
        zscore = (ratio - ratio_train.mean()) / ratio_train.std()
        df_post_features[col_name] = zscore
        print(f"[IMPL CORR] {col_name} ajoutée")

# Corrélation rolling VIX-SPX (proxy direct : corrélation glissante 30j)
if spx_col and spx_col in df_post_features.columns:
    spx_ret_s = df_post_features[spx_col].pct_change()
    vix_spx_corr = vix_ret.rolling(30, min_periods=15).corr(spx_ret_s)
    vix_spx_corr.name = "vix_spx_corr_30d"
    df_post_features["vix_spx_corr_30d"] = vix_spx_corr
    print(f"[IMPL CORR] vix_spx_corr_30d ajoutée "
          f"(train mean={vix_spx_corr.loc[vix_spx_corr.index < pd.Timestamp(TEST_DATE)].mean():.3f})")

# ── 4. Hawkes Process ────────────────────────────────────────────────────────
# Intensité λ_t = μ + Σ α·exp(-β·(t - tᵢ)) sur les sauts passés
# Capte le clustering des chocs (les crises arrivent en clusters)
# Paramètres fixés : α=0.3, β=0.1 (décroissance sur ~10 jours)
HAWKES_ALPHA = 0.3
HAWKES_BETA  = 0.1
HAWKES_THRESHOLD = 2.0  # saut si |r| > 2σ rolling

sigma_hawkes = vix_ret.rolling(30, min_periods=15).std()
jump_times   = vix_ret.index[vix_ret.abs() > HAWKES_THRESHOLD * sigma_hawkes]

hawkes_intensity = pd.Series(0.0, index=vix_ret.index)
for t_idx, t in enumerate(vix_ret.index):
    past_jumps = jump_times[jump_times < t]
    if len(past_jumps) == 0:
        hawkes_intensity.iloc[t_idx] = HAWKES_ALPHA  # μ baseline
        continue
    days_since = np.array([(t - tj).days for tj in past_jumps])
    hawkes_intensity.iloc[t_idx] = HAWKES_ALPHA + HAWKES_ALPHA * np.sum(
        np.exp(-HAWKES_BETA * days_since)
    )

hawkes_intensity.name = "hawkes_intensity"
# Normaliser sur le train
mu_hw = hawkes_intensity.loc[hawkes_intensity.index < pd.Timestamp(TEST_DATE)].mean()
sd_hw = hawkes_intensity.loc[hawkes_intensity.index < pd.Timestamp(TEST_DATE)].std()
hawkes_zscore = (hawkes_intensity - mu_hw) / sd_hw
hawkes_zscore.name = "hawkes_zscore"

print(f"[HAWKES] train mean intensity={mu_hw:.4f}, std={sd_hw:.4f}")
print(f"[HAWKES] nombre de sauts détectés : {len(jump_times)}")

# ── Injection dans df_post_features ─────────────────────────────────────────
new_features = {
    "VRP":                vrp,
    "VRP_zscore":         vrp_zscore,
    "VRP_ma5":            vrp_ma5,
    "jump_intensity_60d": jump_intensity_60d,
    "jump_intensity_20d": jump_intensity_20d,
    "hawkes_intensity":   hawkes_intensity,
    "hawkes_zscore":      hawkes_zscore,
}
for name, series in new_features.items():
    df_post_features[name] = series

added = [f for f in new_features if f in df_post_features.columns]
features_post_engineering = features_post_engineering + added + [
    f for f in ["vix_spx_corr_30d"] +
    [f"impl_corr_proxy_{p.split('_')[0]}" for p,_ in sector_pairs]
    if f in df_post_features.columns and f not in features_post_engineering
]

print(f"\n[MATH FEATURES] {len(added)} features ajoutées → {len(features_post_engineering)} total")


[VRP] train mean=0.042064, std=0.055630
[JUMP] train mean λ_60d=0.0158
[IMPL CORR] vix_spx_corr_30d ajoutée (train mean=-0.803)
[HAWKES] train mean intensity=0.4134, std=0.1369
[HAWKES] nombre de sauts détectés : 381

[MATH FEATURES] 7 features ajoutées → 1020 total


In [27]:
class AmplitudeTargetBuilder:
    def __init__(self, horizon_days=1):
        self.horizon_days = horizon_days
        self.vix_col = None
        self.calm_thr_ = self.stress_thr_ = None
        self.class_thresholds_by_regime_ = {}

    def build(self, df, train_end):
        df = df.copy().sort_index()
        for c in ["VIX_Price","VIX","^VIX"]:
            if c in df.columns: self.vix_col = c; break
        if not self.vix_col: raise ValueError("VIX introuvable")
        vix = pd.to_numeric(df[self.vix_col], errors="coerce")
        vix_tr = vix.loc[vix.index < pd.Timestamp(train_end)]
        self.calm_thr_   = vix_tr.quantile(0.33)
        self.stress_thr_ = vix_tr.quantile(0.67)
        regime = pd.Series("NORMAL", index=df.index)
        regime.loc[vix < self.calm_thr_]    = "CALM"
        regime.loc[vix >= self.stress_thr_] = "STRESS"
        future_vix = vix.shift(-self.horizon_days)
        vix_return = (future_vix / vix) - 1
        flat_mask = vix_return.abs() < FLAT_THRESHOLD_ABS
        df["VIX_Return"] = vix_return
        df["VIX_Regime"] = regime
        df = df.loc[~flat_mask].dropna(subset=["VIX_Return"])
        print(f"[TARGET h={self.horizon_days}j] {flat_mask.sum()} flat supprimés, {len(df)} lignes")
        df_train = df.loc[df.index < pd.Timestamp(train_end)]
        for reg in ["CALM","NORMAL","STRESS"]:
            sub = df_train.loc[df_train["VIX_Regime"]==reg, "VIX_Return"]
            q25 = sub.quantile(CLASS_QUANTILES[0]) if len(sub)>=20 else 0
            q75 = sub.quantile(CLASS_QUANTILES[1]) if len(sub)>=20 else 0
            self.class_thresholds_by_regime_[reg] = (q25, q75)
        q25g = df_train["VIX_Return"].quantile(CLASS_QUANTILES[0])
        q75g = df_train["VIX_Return"].quantile(CLASS_QUANTILES[1])
        self.class_thresholds_by_regime_["GLOBAL"] = (q25g, q75g)
        print(f"  Seuils GLOBAL: q25={q25g:.3%} q75={q75g:.3%}")
        def classify(row):
            q25, q75 = self.class_thresholds_by_regime_.get(row["VIX_Regime"],(0,0))
            r = row["VIX_Return"]
            if r < q25: return "DOWN_FORT"
            if r < 0:   return "DOWN_FAIBLE"
            if r < q75: return "UP_FAIBLE"
            return "UP_FORT"
        df["VIX_Amplitude_Class"] = df.apply(classify, axis=1)
        print(df["VIX_Amplitude_Class"].value_counts().sort_index())
        return df


In [28]:
def clf_configs():
    return {
        "XGBoost": (XGBClassifier,
            {"max_depth":[3,5],"learning_rate":[0.03,0.05],
             "n_estimators":[150,250],"subsample":[0.8],
             "colsample_bytree":[0.7,0.8],"min_child_weight":[1,3]},
            {"random_state":RANDOM_STATE,"eval_metric":"mlogloss",
             "objective":"multi:softprob","n_jobs":-1}),
        "LightGBM": (LGBMClassifier,
            {"num_leaves":[15,31],"learning_rate":[0.03,0.05],
             "n_estimators":[150,250],"max_depth":[4,6],
             "min_child_samples":[5,10]},
            {"random_state":RANDOM_STATE,"verbose":-1,"class_weight":"balanced"}),
        "GradientBoosting": (GradientBoostingClassifier,
            {"n_estimators":[150,250],"learning_rate":[0.03,0.05],
             "max_depth":[3,5],"min_samples_leaf":[5,10],"subsample":[0.8]},
            {"random_state":RANDOM_STATE}),
        "RandomForest": (RandomForestClassifier,
            {"n_estimators":[200],"max_depth":[5,8],"min_samples_leaf":[5,10]},
            {"random_state":RANDOM_STATE,"n_jobs":-1,"class_weight":"balanced"}),
        "LogisticRegression": (LogisticRegression,
            {"C":[0.1,1.0,10.0]},
            {"random_state":RANDOM_STATE,"max_iter":2000,
             "class_weight":"balanced","multi_class":"multinomial","solver":"lbfgs"}),
    }

def compute_hierarchical_metrics(y_true, y_pred):
    dir_map = {"DOWN_FORT":"DOWN","DOWN_FAIBLE":"DOWN","UP_FAIBLE":"UP","UP_FORT":"UP"}
    yd_t = [dir_map.get(y,"DOWN") for y in y_true]
    yd_p = [dir_map.get(y,"DOWN") for y in y_pred]
    acc_dir = accuracy_score(yd_t, yd_p)
    f1_up = f1_score(yd_t,yd_p,pos_label="UP",  average="binary",zero_division=0)
    f1_dn = f1_score(yd_t,yd_p,pos_label="DOWN",average="binary",zero_division=0)
    p_up  = precision_score(yd_t,yd_p,pos_label="UP",  average="binary",zero_division=0)
    r_up  = recall_score(yd_t,yd_p,pos_label="UP",     average="binary",zero_division=0)
    p_dn  = precision_score(yd_t,yd_p,pos_label="DOWN",average="binary",zero_division=0)
    r_dn  = recall_score(yd_t,yd_p,pos_label="DOWN",   average="binary",zero_division=0)

    def sub_f1(idx, fort_key):
        if len(idx) < 10: return None, None, None
        yt = ["FORT" if y_true[i]==fort_key else "FAIBLE" for i in idx]
        yp = ["FORT" if y_pred[i]==fort_key else "FAIBLE" for i in idx]
        return (f1_score(yt,yp,pos_label="FORT",  average="binary",zero_division=0),
                f1_score(yt,yp,pos_label="FAIBLE",average="binary",zero_division=0),
                accuracy_score(yt,yp))

    up_idx = [i for i,y in enumerate(y_true) if dir_map.get(y)=="UP"]
    dn_idx = [i for i,y in enumerate(y_true) if dir_map.get(y)=="DOWN"]
    f1_uf,f1_uf2,acc_up = sub_f1(up_idx,"UP_FORT")
    f1_df,f1_df2,acc_dn = sub_f1(dn_idx,"DOWN_FORT")

    def r(v): return round(v,4) if v is not None else None
    return {
        "F1_4cls":        r(f1_score(y_true,y_pred,average="macro",labels=CLASS_LABELS,zero_division=0)),
        "Acc_4cls":       r(accuracy_score(y_true,y_pred)),
        "Acc_dir":        r(acc_dir),
        "F1_dir":         r((f1_up+f1_dn)/2),
        "F1_UP":r(f1_up), "Prec_UP":r(p_up), "Rec_UP":r(r_up),
        "F1_DOWN":r(f1_dn),"Prec_DOWN":r(p_dn),"Rec_DOWN":r(r_dn),
        "F1_UP_FORT":r(f1_uf),"F1_UP_FAIBLE":r(f1_uf2),"Acc_UP_sub":r(acc_up),
        "F1_DOWN_FORT":r(f1_df),"F1_DOWN_FAIBLE":r(f1_df2),"Acc_DOWN_sub":r(acc_dn),
    }


In [29]:
def generate_all_interactions(df, base_features, top_n=20, rolling_w=20, eps=1e-8):
    feats = [f for f in base_features[:top_n] if f in df.columns]
    new_cols = {}
    for i in range(len(feats)):
        for j in range(i+1, len(feats)):
            fi,fj = feats[i],feats[j]
            si,sj = df[fi],df[fj]
            sd = sj.where(sj.abs()>=eps, np.nan)
            new_cols[f"{fi}__div__{fj}"]     = si/sd
            new_cols[f"{fi}__minus__{fj}"]   = si-sj
            new_cols[f"{fi}__prod__{fj}"]     = si*sj
            diff = si-sj
            rs = diff.rolling(rolling_w,min_periods=rolling_w//2).std()
            new_cols[f"{fi}__zrel__{fj}"]    = diff/rs.replace(0,np.nan)
            ma_i = si.rolling(rolling_w,min_periods=rolling_w//2).mean()
            ma_j = sj.rolling(rolling_w,min_periods=rolling_w//2).mean()
            new_cols[f"{fi}__macross__{fj}"] = ma_i/ma_j.where(ma_j.abs()>=eps,np.nan)
            new_cols[f"{fi}__ret5x__{fj}"]   = si.pct_change(5)*sj
    idf = pd.DataFrame(new_cols,index=df.index).replace([np.inf,-np.inf],np.nan)
    idf = idf.dropna(axis=1,how="all")
    print(f"[INTER] {len(feats)}f → {idf.shape[1]} colonnes")
    return idf


def shap_select_features(X_train, y_train, top_n, label=""):
    le = LabelEncoder(); le.fit(CLASS_LABELS)
    pilot = XGBClassifier(n_estimators=SHAP_PILOT_N_ESTIMATORS,
                          max_depth=3, learning_rate=0.05, subsample=0.8,
                          eval_metric="mlogloss", objective="multi:softprob",
                          random_state=RANDOM_STATE, n_jobs=-1)
    pilot.fit(X_train.values, le.transform(y_train))
    expl = shap.TreeExplainer(pilot)
    sv   = expl.shap_values(X_train.values)
    if isinstance(sv,list):
        arr = np.mean([np.abs(s) for s in sv],axis=0)
    elif np.array(sv).ndim==3:
        arr = np.abs(sv).mean(axis=2)
    else:
        arr = np.abs(sv)
    scores = pd.Series(arr.mean(axis=0), index=X_train.columns)
    top = scores.sort_values(ascending=False).head(top_n).index.tolist()
    if label: print(f"[SHAP {label}] top-{top_n}/{len(X_train.columns)}")
    return top


In [30]:
_SEPS = ["__div__","__minus__","__prod__","__zrel__","__macross__","__ret5x__"]
phase1 = {}

for h in HORIZONS_TO_TEST:
    print(f"\n{'#'*60}\n# PHASE 1 SHAP | h={h}j\n{'#'*60}")
    phase1[h] = {}

    atb = AmplitudeTargetBuilder(horizon_days=h)
    df_full = atb.build(df_post_features, train_end=TEST_DATE)
    df_tr_full = df_full.loc[df_full.index < pd.Timestamp(TEST_DATE)]
    feats_avail = [f for f in features_post_engineering if f in df_tr_full.columns]

    for regime in ALL_REGIMES:
        print(f"\n--- {regime} ---")
        df_tr = df_tr_full if regime=="GLOBAL" else df_tr_full[df_tr_full["VIX_Regime"]==regime]
        if len(df_tr)<80 or df_tr["VIX_Amplitude_Class"].nunique()<2:
            print("[WARN] Skip."); continue

        cleaner = TrainFittedCleaner()
        X_b = pd.DataFrame(cleaner.fit_transform(df_tr[feats_avail]),
                            columns=feats_avail,index=df_tr.index).dropna(axis=1,how="all")
        X_b_sc = pd.DataFrame(StandardScaler().fit_transform(X_b),columns=X_b.columns,index=X_b.index)
        y_b = df_tr["VIX_Amplitude_Class"].loc[X_b_sc.index].values

        top_base  = shap_select_features(X_b_sc, y_b, SHAP_TOP_BASE_N, f"{regime} base")
        idf       = generate_all_interactions(df_tr[top_base], top_base, N_TOP_FOR_INTERACTIONS, INTERACTION_ROLLING_WINDOW)
        extended  = top_base + list(idf.columns)
        df_ext    = pd.concat([df_tr[top_base],idf],axis=1)
        df_ext    = df_ext.loc[:,~df_ext.columns.duplicated()]
        cleaner2  = TrainFittedCleaner()
        X_e = pd.DataFrame(cleaner2.fit_transform(df_ext[extended]),
                            columns=extended,index=df_ext.index).dropna(axis=1,how="all")
        X_e_sc = pd.DataFrame(StandardScaler().fit_transform(X_e),columns=X_e.columns,index=X_e.index)
        y_e    = df_tr["VIX_Amplitude_Class"].loc[X_e_sc.index].values
        top_final = shap_select_features(X_e_sc, y_e, SHAP_TOP_FINAL_N, f"{regime} +inter")

        n_inter = sum(1 for f in top_final if any(s in f for s in _SEPS))
        n_ts    = sum(1 for f in top_final if any(k in f for k in
                      ["VIX_Residual","VIX_Innovation","P_stress_HMM","EGARCH_SPX"]))
        print(f"[FINAL] {len(top_final)}f ({n_inter} inter, {n_ts} TS/EGARCH)")

        # TRAIN_START auto
        base_in = [f for f in top_final if f in df_full.columns]
        inter_n = [f for f in top_final if f not in df_full.columns]
        src_set = set(base_in)
        for fname in inter_n:
            for sep in _SEPS:
                if sep in fname:
                    a,b = fname.split(sep,1)
                    if a in df_full.columns: src_set.add(a)
                    if b in df_full.columns: src_set.add(b)
                    break
        sl = [f for f in src_set if f in df_full.columns]
        if inter_n and sl:
            idf2  = generate_all_interactions(df_full[sl],sl,len(sl),INTERACTION_ROLLING_WINDOW)
            avail = [f for f in inter_n if f in idf2.columns]
            df_st = pd.concat([df_full[base_in],idf2[avail]],axis=1)
        else:
            df_st = df_full[base_in]

        fv = df_st.dropna(how="any").index.min()
        ts = (fv if not pd.isna(fv) else pd.Timestamp(TRAIN_START)).strftime("%Y-%m-%d")
        print(f"[TRAIN_START] {ts}")
        phase1[h][regime] = {"features":top_final,"train_start":ts,
                              "n_interactions":n_inter,"n_ts":n_ts,"df_full":df_full}

pd.DataFrame([{"H":h,"Regime":r,"Train_Start":v["train_start"],
               "N_Feat":len(v["features"]),"N_Inter":v["n_interactions"],"N_TS":v["n_ts"]}
              for h,regimes in phase1.items() for r,v in regimes.items()]
             ).to_csv(OUTPUT_DIR/"phase1_shap.csv",index=False)
print("\n[SAVE] phase1_shap.csv")



############################################################
# PHASE 1 SHAP | h=1j
############################################################
[TARGET h=1j] 511 flat supprimés, 6226 lignes
  Seuils GLOBAL: q25=-4.073% q75=3.610%
VIX_Amplitude_Class
DOWN_FAIBLE    1832
DOWN_FORT      1550
UP_FAIBLE      1271
UP_FORT        1573
Name: count, dtype: int64

--- CALM ---
[SHAP CALM base] top-40/1020
[INTER] 20f → 1140 colonnes
[SHAP CALM +inter] top-30/1180
[FINAL] 30f (19 inter, 3 TS/EGARCH)
[INTER] 27f → 2106 colonnes
[TRAIN_START] 2001-08-01

--- NORMAL ---
[SHAP NORMAL base] top-40/1020
[INTER] 20f → 1140 colonnes
[SHAP NORMAL +inter] top-30/1180
[FINAL] 30f (24 inter, 6 TS/EGARCH)
[INTER] 24f → 1656 colonnes
[TRAIN_START] 2000-11-15

--- STRESS ---
[SHAP STRESS base] top-40/1020
[INTER] 20f → 1140 colonnes
[SHAP STRESS +inter] top-30/1180
[FINAL] 30f (24 inter, 2 TS/EGARCH)
[INTER] 23f → 1518 colonnes
[TRAIN_START] 2000-11-02

--- GLOBAL ---
[SHAP GLOBAL base] top-40/1020
[INTER] 20f

In [ ]:
print("="*60)
print("PHASE 2 — 3 horizons | flat supprimés | EGARCH SPX features")
print("="*60)

clf_rows = []
model_num = 0

for h in HORIZONS_TO_TEST:
    for regime in ALL_REGIMES:
        info = phase1[h].get(regime)
        if not info: continue

        top_final   = info["features"]
        train_start = info["train_start"]
        df_full     = info["df_full"]
        print(f"\n[h={h}j | {regime}] {len(top_final)}f | train_start={train_start}")

        df_w = df_full.loc[df_full.index >= pd.Timestamp(train_start)].copy()

        # Reconstruire interactions
        base_in = [f for f in top_final if f in df_w.columns]
        inter_n = [f for f in top_final if f not in df_w.columns]
        if inter_n:
            src = set(base_in)
            for fname in inter_n:
                for sep in _SEPS:
                    if sep in fname:
                        a,b = fname.split(sep,1)
                        if a in df_w.columns: src.add(a)
                        if b in df_w.columns: src.add(b)
                        break
            sl = [f for f in src if f in df_w.columns]
            if sl:
                idf2 = generate_all_interactions(df_w[sl],sl,len(sl),INTERACTION_ROLLING_WINDOW)
                avail = [f for f in inter_n if f in idf2.columns]
                df_w  = pd.concat([df_w,idf2[avail]],axis=1)

        feats_run = [f for f in top_final if f in df_w.columns]
        df_w[feats_run] = df_w[feats_run].replace([np.inf,-np.inf],np.nan)

        dtr = df_w.loc[df_w.index < pd.Timestamp(TEST_DATE)]
        dte = df_w.loc[df_w.index >= pd.Timestamp(TEST_DATE)]
        if regime != "GLOBAL":
            dtr = dtr.loc[dtr["VIX_Regime"]==regime]
            dte = dte.loc[dte["VIX_Regime"]==regime]

        if len(dtr)<80 or len(dte)<20:
            print(f"[WARN] train={len(dtr)} test={len(dte)}. Skip."); continue

        y_tr = dtr["VIX_Amplitude_Class"].values
        y_te = dte["VIX_Amplitude_Class"].values
        if len(np.unique(y_tr))<2: print("[WARN] 1 classe. Skip."); continue

        cleaner = TrainFittedCleaner()
        X_tr_c = cleaner.fit_transform(dtr[feats_run])
        X_te_c = cleaner.transform(dte[feats_run])
        sc = StandardScaler()
        X_tr_s = pd.DataFrame(sc.fit_transform(X_tr_c),columns=feats_run,index=dtr.index)
        X_te_s = pd.DataFrame(sc.transform(X_te_c),   columns=feats_run,index=dte.index)

        le = LabelEncoder(); le.fit(CLASS_LABELS)
        y_enc = le.transform(y_tr)
        try:
            sm = SMOTE(random_state=RANDOM_STATE)
            X_arr,y_sm = sm.fit_resample(X_tr_s.values,y_enc)
            X_tr_fit = pd.DataFrame(X_arr,columns=feats_run); y_fit = y_sm
        except Exception:
            X_tr_fit = X_tr_s; y_fit = y_enc

        max_n = min(MAX_N_FEATURES, len(feats_run))
        cv    = TimeSeriesSplit(n_splits=3)

        for algo_name,(Cls,pgrid,fixed) in clf_configs().items():
            for n in range(MIN_N_FEATURES, max_n+1):
                fn  = feats_run[:n]
                Xtr = X_tr_fit[fn].values if isinstance(X_tr_fit,pd.DataFrame) else X_tr_fit[:,:n]
                Xte = X_te_s[fn].values
                try:
                    grid = GridSearchCV(Cls(**fixed),pgrid,scoring="f1_macro",cv=cv,n_jobs=-1)
                    grid.fit(Xtr,y_fit)
                    model = grid.best_estimator_
                except Exception: continue
                y_pred_enc = model.predict(Xte)
                y_pred_str = le.inverse_transform(y_pred_enc)
                m = compute_hierarchical_metrics(list(y_te),list(y_pred_str))
                model_num += 1
                clf_rows.append({"Model_Number":model_num,"Horizon":h,"Regime":regime,
                                 "Algo":algo_name,"N_Features":n,
                                 "Train_Start":train_start,"Features":json.dumps(fn),**m})
            print(f"  {algo_name} ✓")

clf_df = pd.DataFrame(clf_rows)
clf_df.to_csv(OUTPUT_DIR/"phase2_results.csv",index=False)
print(f"\n[DONE] {len(clf_df)} modèles")


PHASE 2 — 3 horizons | flat supprimés | EGARCH SPX features

[h=1j | CALM] 30f | train_start=2001-08-01
[INTER] 27f → 2106 colonnes
  XGBoost ✓
  LightGBM ✓
  GradientBoosting ✓
  RandomForest ✓
  LogisticRegression ✓

[h=1j | NORMAL] 30f | train_start=2000-11-15
[INTER] 24f → 1656 colonnes
  XGBoost ✓
  LightGBM ✓
  GradientBoosting ✓
  RandomForest ✓
  LogisticRegression ✓

[h=1j | STRESS] 30f | train_start=2000-11-02
[INTER] 23f → 1518 colonnes
  XGBoost ✓
  LightGBM ✓
  GradientBoosting ✓
  RandomForest ✓
  LogisticRegression ✓

[h=1j | GLOBAL] 30f | train_start=2000-09-08
[INTER] 13f → 468 colonnes
  XGBoost ✓
  LightGBM ✓
  GradientBoosting ✓
  RandomForest ✓
  LogisticRegression ✓

[h=3j | CALM] 30f | train_start=2001-08-01
[INTER] 25f → 1800 colonnes
  XGBoost ✓
  LightGBM ✓
  GradientBoosting ✓
  RandomForest ✓
  LogisticRegression ✓

[h=3j | NORMAL] 30f | train_start=2000-11-15
[INTER] 25f → 1800 colonnes
  XGBoost ✓
  LightGBM ✓
  GradientBoosting ✓
  RandomForest ✓
  Logist

In [1]:
print("="*60)
print("SYNTHÈSE | seuils: F1_dir>0.5, F1_FORT>0.5")
print("="*60)

best_rows = []
for h in HORIZONS_TO_TEST:
    print(f"\n{'='*50}\nHORIZON = {h}j")
    for regime in ALL_REGIMES:
        sub = clf_df[(clf_df["Horizon"]==h) & (clf_df["Regime"]==regime)]
        if sub.empty: continue
        best = sub.loc[sub["F1_dir"].idxmax()]
        ok_dir = "✓" if best["F1_dir"]>=0.5 else "✗"
        ok_uf  = "✓" if (best["F1_UP_FORT"] or 0)>=0.5 else "✗"
        ok_df  = "✓" if (best["F1_DOWN_FORT"] or 0)>=0.5 else "✗"
        print(f"""
{regime} | {best["Algo"]} N={best["N_Features"]}
  [L1] Acc={best["Acc_dir"]}  F1_dir={best["F1_dir"]} {ok_dir}
       F1_UP={best["F1_UP"]} P={best["Prec_UP"]} R={best["Rec_UP"]}
       F1_DOWN={best["F1_DOWN"]} P={best["Prec_DOWN"]} R={best["Rec_DOWN"]}
  [L2] F1_UP_FORT={best["F1_UP_FORT"]} {ok_uf}  F1_UP_FAIBLE={best["F1_UP_FAIBLE"]}
  [L3] F1_DOWN_FORT={best["F1_DOWN_FORT"]} {ok_df}  F1_DOWN_FAIBLE={best["F1_DOWN_FAIBLE"]}""")
        best_rows.append({**{"Horizon":h,"Regime":regime},**best.to_dict()})

best_df = pd.DataFrame(best_rows)
with pd.ExcelWriter(OUTPUT_DIR/"vix_egarch_spx_report.xlsx",engine="xlsxwriter") as w:
    pd.read_csv(OUTPUT_DIR/"phase1_shap.csv").to_excel(w,"Phase1_SHAP",index=False)
    clf_df.to_excel(w,"All_Results",index=False)
    best_df.to_excel(w,"Best_Per_Horizon_Regime",index=False)
print(f"\n[SAVE] vix_egarch_spx_report.xlsx")
print("[NOTE] Aucun modèle enregistré.")


SYNTHÈSE | seuils: F1_dir>0.5, F1_FORT>0.5


NameError: name 'HORIZONS_TO_TEST' is not defined

In [ ]:
from google.colab import files

# Download the main report file
files.download(OUTPUT_DIR / "vix_egarch_spx_report.xlsx")

# Download the phase1 SHAP results file
files.download(OUTPUT_DIR / "phase1_shap.csv")

# You might also want to download the detailed results if needed
files.download(OUTPUT_DIR / "phase2_results.csv")